In [ ]:
from scipy.interpolate import griddata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import csv
from matplotlib.lines import Line2D
from scipy.integrate import quad
from functools import reduce
import operator
Cww = 1
Cgg = 0
Cbb = -1
mz = 91.188
mw = 80.3692
cw = mw/mz
sw = np.sqrt(1 - cw**2) 
alpha = 1/128
alpha_s = 0.11 #mz
hbar_c = 1.973e-14  # GeV·cm

Caa = Cww + Cbb
Caz = cw**2 * Cww - sw**2 * Cbb
Czz = cw**4 * Cww + sw**4 * Cbb

In [ ]:
L1 = 25 #3025
L2 = 150 #ab-1 #7407

In [ ]:
ma = [10, 30, 50, 70, 85]
S_eff = [0.62, 0.768, 0.732, 0.581, 0.162]
B_eff = [0.035, 0.035, 0.035, 0.035, 0.035]
B_cs = 84.29*0.178*0.174
mtau2=1.777*2
# 读取数据
pho_ALP = pd.read_csv('./alp_decay_widths_had2.csv')
# 将数据类型转换为更小的浮动数类型以减少内存使用
pho_ALP_tautau = 0.178*0.174*pho_ALP['xs_tautau_prompt'].astype('float32')
pho_ALP_tautau1 = 0.178*0.174*pho_ALP['xs_tautau'].astype('float32')
pho_ALP_lambda = 1000 / pho_ALP['Lambda']

xs = pho_ALP_tautau
xs1 = pho_ALP_tautau1
gstar = pho_ALP_lambda
MQ = pho_ALP['ma']
# 为信号截面插值，生成从7到79GeV每隔2GeV的数据点
signal_xs_interp = np.interp(pho_ALP['ma'], ma, S_eff)

In [ ]:
# 创建网格，减少网格密度以降低内存使用
mq_14_os = np.logspace(np.log10(pho_ALP['ma'].min()), np.log10(pho_ALP['ma'].max()), 2000)
gstar_14_os = np.logspace(np.log10(gstar.min()), np.log10(gstar.max()), 200)
#mq_14_os = np.linspace(pho_ALP['ma'].min(), pho_ALP['ma'].max(), 2000)
#gstar_14_os = np.linspace(gstar.min(), gstar.max(), 200)
# 创建网格
MQ_grid_14_os, Gstar_grid_14_os = np.meshgrid(mq_14_os, gstar_14_os)

# 构造插值用的点和目标点
points_frac = np.vstack((MQ, gstar)).T

# 使用二维插值来计算网格上的 S 和 Z
points_os = MQ,gstar.values
values_os = xs.values
values_os1 = xs1.values

# 对每个点进行信号和背景效率的插值
signal_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, S_eff)
background_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, B_eff)

# 将效率的插值结果重塑为(100, 100)形状
signal_efficiency_grid = signal_efficiency.reshape(MQ_grid_14_os.shape)
background_efficiency_grid = background_efficiency.reshape(MQ_grid_14_os.shape)


# 重新计算S_grid和背景
S_grid_14_os_emu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')* L1*1000 * signal_efficiency_grid * 1000  # 25fb
background_14_os_emu = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000  # 25fb

# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_emu = S_grid_14_os_emu / np.sqrt(S_grid_14_os_emu + background_14_os_emu)

# 创建30fb情况
# 用30fb进行计算
S_grid_14_os_30_emu = griddata(points_os, values_os1, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * signal_efficiency_grid * 1000  
background_14_os_30_emu = griddata(points_os, tuple([B_cs]*np.shape(values_os1)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000 


# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_30_emu = S_grid_14_os_30_emu / np.sqrt(S_grid_14_os_30_emu + background_14_os_30_emu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)

std_3_emu=sigma_Z(S_grid_14_os_emu,background_14_os_emu)
#Z_3_amu= Z_grid_14_os_amu / np.sqrt(std_3_amu)
z3emup = Z_grid_14_os_emu+np.sqrt(std_3_emu)
z3emum = Z_grid_14_os_emu-np.sqrt(std_3_emu)

std_30_emu=sigma_Z(S_grid_14_os_30_emu,background_14_os_30_emu)
#Z_30_amu= Z_grid_14_os_30_amu / np.sqrt(std_30_amu)
z30emup = Z_grid_14_os_30_emu+np.sqrt(std_30_emu)
z30emum = Z_grid_14_os_30_emu-np.sqrt(std_30_emu)

# 绘制单张图包含两个等值线图
plt.figure(figsize=(8, 6))

# Z_grid_14_os_30
contour1 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_30_emu, levels=[2], colors='darkorange')
#plt.clabel(contour1, fmt={2: fmt2}, inline=True, fontsize=10)
contour_emu30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30emup, levels=[2], colors='darkorange',linestyles='--')
contour_emu30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30emum, levels=[2], colors='darkorange',linestyles='--')
upper_segments_30 = contour_emu30p.allsegs[0]
lower_segments_30 = contour_emu30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#ffc680', alpha=0.5)

# Z_grid_14_os
contour2 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_emu, levels=[2], colors='navy')
#plt.clabel(contour2, fmt={2: fmt1}, inline=True, fontsize=10,manual=[(70,2e-3)])
contour_emu3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3emup, levels=[2], colors='navy',linestyles='--')
contour_emu3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3emum, levels=[2], colors='navy',linestyles='--')
upper_segments_3 = contour_emu3p.allsegs[0]
lower_segments_3 = contour_emu3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#8080c0', alpha=0.5)

#plt.title(r'ALP$\rightarrow \tau\tau \rightarrow e \mu $')
plt.ylim(1e-2,1e1)
plt.xlim(1,92)
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.axvline(x=mtau2, linewidth=1.5)
plt.yscale('log')
plt.xscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label=R'2$\sigma$ C.L. 25 ab$^{-1}$'),
    Line2D([0], [0], color='darkorange', lw=2, label=R'2$\sigma$ C.L. 150 ab$^{-1}$ '),
    Line2D([0], [0], color='navy', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='darkorange', lw=2, linestyle='--', label=R'150 ab$^{-1}$ 1$\sigma$ Error boundary')
]
plt.text(0.6, 0.8, R'$a \to \tau^{+}(\tau^{+}\to \mu^{+} \bar{v}_{\tau} v_{\mu}) \tau^{-}(\tau^{-}\to e^{-} v_{\tau} \bar{v}_{e})$', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
plt.legend(handles=legend_elements, loc='best', fontsize=10)
plt.savefig('./prompt/tauemu.png',dpi=300)
plt.show()

In [ ]:
ma = [10, 30, 50, 70, 85]
S_eff = [0.921, 0.807, 0.756, 0.599, 0.234]
B_eff = [0.04, 0.04, 0.04, 0.04, 0.04]
B_cs = 84.29*0.174**2

# 读取数据
pho_ALP = pd.read_csv('./alp_decay_widths_had2.csv')

# 将数据类型转换为更小的浮动数类型以减少内存使用
pho_ALP_tautau = 0.174*0.174*pho_ALP['xs_tautau_prompt'].astype('float32')
pho_ALP_tautau1 = 0.174*0.174*pho_ALP['xs_tautau'].astype('float32')
pho_ALP_lambda = 1000 / pho_ALP['Lambda']

xs = pho_ALP_tautau
xs1 = pho_ALP_tautau1
gstar = pho_ALP_lambda
MQ = pho_ALP['ma']
# 为信号截面插值，生成从7到79GeV每隔2GeV的数据点
signal_xs_interp = np.interp(pho_ALP['ma'], ma, S_eff)

# 创建网格，减少网格密度以降低内存使用
mq_14_os = np.logspace(np.log10(pho_ALP['ma'].min()), np.log10(pho_ALP['ma'].max()), 2000)
gstar_14_os = np.logspace(np.log10(gstar.min()), np.log10(gstar.max()), 200)

# 创建网格
MQ_grid_14_os, Gstar_grid_14_os = np.meshgrid(mq_14_os, gstar_14_os)


# 使用二维插值来计算网格上的 S 和 Z
points_os = MQ,gstar.values
values_os = xs.values
values_os1 = xs1.values

# 对每个点进行信号和背景效率的插值
signal_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, S_eff)
background_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, B_eff)

# 将效率的插值结果重塑为(100, 100)形状
signal_efficiency_grid = signal_efficiency.reshape(MQ_grid_14_os.shape)
background_efficiency_grid = background_efficiency.reshape(MQ_grid_14_os.shape)


# 重新计算S_grid和背景
S_grid_14_os_e = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * signal_efficiency_grid * 1000  # 25fb
background_14_os_e = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000  # 25fb

# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_e = S_grid_14_os_e / np.sqrt(S_grid_14_os_e + background_14_os_e)

# 创建30fb情况
# 用30fb进行计算
S_grid_14_os_30_e = griddata(points_os, values_os1, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * signal_efficiency_grid * 1000  
background_14_os_30_e = griddata(points_os, tuple([B_cs]*np.shape(values_os1)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000 


# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_30_e = S_grid_14_os_30_e / np.sqrt(S_grid_14_os_30_e + background_14_os_30_e)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)

std_3_e=sigma_Z(S_grid_14_os_e,background_14_os_e)
#Z_3_amu= Z_grid_14_os_amu / np.sqrt(std_3_amu)
z3ep = Z_grid_14_os_e+np.sqrt(std_3_e)
z3em = Z_grid_14_os_e-np.sqrt(std_3_e)

std_30_e=sigma_Z(S_grid_14_os_30_e,background_14_os_30_e)
#Z_30_amu= Z_grid_14_os_30_amu / np.sqrt(std_30_amu)
z30ep = Z_grid_14_os_30_e+np.sqrt(std_30_e)
z30em = Z_grid_14_os_30_e-np.sqrt(std_30_e)

# 绘制单张图包含两个等值线图
plt.figure(figsize=(8, 6))

# Signal yield
N_prompt_150 = xs  * 150 * 10**(6)  # pb \to ab
N_prompt_25 = xs * 25 * 10**(6) # pb \to ab
# Plot settings
labels = [R'Prompt: 150 ab$^{-1}$', R'Prompt: 25 ab$^{-1}$']
handles = []

plt.figure(figsize=(8,6))
N_prompt_150_grid = griddata(points_os, N_prompt_150.values, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')
N_prompt_25_grid = griddata(points_os, N_prompt_25.values, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')



# Z_grid_14_os_30
contour1 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_30_e, levels=[2], colors='darkorange')
#plt.clabel(contour1, fmt={2: fmt2}, inline=True, fontsize=10)
contour_e30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30ep, levels=[2], colors='darkorange',linestyles='--')
contour_e30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30em, levels=[2], colors='darkorange',linestyles='--')
upper_segments_30 = contour_e30p.allsegs[0]
lower_segments_30 = contour_e30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#ffc680', alpha=0.5)

# Z_grid_14_os
contour2 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_e, levels=[2], colors='navy')
#plt.clabel(contour2, fmt={2: fmt1}, inline=True, fontsize=10,manual=[(70,2e-3)])
contour_e3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3ep, levels=[2], colors='navy',linestyles='--')
contour_e3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3em, levels=[2], colors='navy',linestyles='--')
upper_segments_3 = contour_e3p.allsegs[0]
lower_segments_3 = contour_e3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#8080c0', alpha=0.5)

#plt.title(r'ALP$\rightarrow \tau\tau \rightarrow e \mu $')
plt.ylim(1e-2,1e1)
plt.xlim(1e0,92)
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.axvline(x=mtau2, linewidth=1.5)
plt.yscale('log')
plt.xscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label=R'2$\sigma$ C.L. 25 ab$^{-1}$'),
    Line2D([0], [0], color='darkorange', lw=2, label=R'2$\sigma$ C.L. 150 ab$^{-1}$ '),
    Line2D([0], [0], color='navy', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='darkorange', lw=2, linestyle='--', label=R'150 ab$^{-1}$ 1$\sigma$ Error boundary')
]
plt.text(0.6, 0.7, R'$a \to \tau^{+}(\tau^{+}\to e^{+} \bar{v}_{\tau} v_{e}) \tau^{-}(\tau^{-}\to e^{-} v_{\tau} \bar{v}_{e})$', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
plt.legend(handles=legend_elements, loc='best', fontsize=10)
plt.savefig('./prompt/tauee.png',dpi=300)
plt.show()

In [ ]:
ma = [10, 30, 50, 70, 85]
S_eff = [0.923, 0.797, 0.755, 0.623, 0.239]
B_eff = [0.04, 0.04, 0.04, 0.04, 0.04]
B_cs = 84.29*0.178**2

# 读取数据
pho_ALP = pd.read_csv('./alp_decay_widths_had2.csv')

# 将数据类型转换为更小的浮动数类型以减少内存使用
pho_ALP_tautau = 0.178*0.178*pho_ALP['xs_tautau_prompt'].astype('float32')
pho_ALP_tautau1 = 0.178*0.178*pho_ALP['xs_tautau'].astype('float32')
pho_ALP_lambda = 1000 / pho_ALP['Lambda']

xs = pho_ALP_tautau
xs1 = pho_ALP_tautau1
gstar = pho_ALP_lambda
MQ = pho_ALP['ma']
# 为信号截面插值，生成从7到79GeV每隔2GeV的数据点
signal_xs_interp = np.interp(pho_ALP['ma'], ma, S_eff)

# 创建网格，减少网格密度以降低内存使用
mq_14_os = np.logspace(np.log10(pho_ALP['ma'].min()), np.log10(pho_ALP['ma'].max()), 2000)
gstar_14_os = np.logspace(np.log10(gstar.min()), np.log10(gstar.max()), 200)

# 创建网格
MQ_grid_14_os, Gstar_grid_14_os = np.meshgrid(mq_14_os, gstar_14_os)

# 使用二维插值来计算网格上的 S 和 Z
points_os = MQ,gstar.values
values_os = xs.values
values_os1 = xs1.values

# 对每个点进行信号和背景效率的插值
signal_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, S_eff)
background_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, B_eff)

# 将效率的插值结果重塑为(100, 100)形状
signal_efficiency_grid = signal_efficiency.reshape(MQ_grid_14_os.shape)
background_efficiency_grid = background_efficiency.reshape(MQ_grid_14_os.shape)


# 重新计算S_grid和背景
S_grid_14_os_mu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * signal_efficiency_grid * 1000  # 25fb
background_14_os_mu = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000  # 25fb

# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_mu = S_grid_14_os_mu / np.sqrt(S_grid_14_os_mu + background_14_os_mu)

# 创建30fb情况
# 用30fb进行计算
S_grid_14_os_30_mu = griddata(points_os, values_os1, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * signal_efficiency_grid * 1000  
background_14_os_30_mu = griddata(points_os, tuple([B_cs]*np.shape(values_os1)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000 


# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_30_mu = S_grid_14_os_30_mu / np.sqrt(S_grid_14_os_30_mu + background_14_os_30_mu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)

std_3_mu=sigma_Z(S_grid_14_os_mu,background_14_os_mu)
#Z_3_amu= Z_grid_14_os_amu / np.sqrt(std_3_amu)
z3mup = Z_grid_14_os_mu+np.sqrt(std_3_mu)
z3mum = Z_grid_14_os_mu-np.sqrt(std_3_mu)

std_30_mu=sigma_Z(S_grid_14_os_30_mu,background_14_os_30_mu)
#Z_30_amu= Z_grid_14_os_30_amu / np.sqrt(std_30_amu)
z30mup = Z_grid_14_os_30_mu+np.sqrt(std_30_mu)
z30mum = Z_grid_14_os_30_mu-np.sqrt(std_30_mu)

# 绘制单张图包含两个等值线图
plt.figure(figsize=(8, 6))

# Signal yield
N_prompt_150 = xs  * 150 * 10**(6)  # pb \to ab
N_prompt_25 = xs * 25 * 10**(6) # pb \to ab
# Plot settings
labels = [R'Prompt: 150 ab$^{-1}$', R'Prompt: 25 ab$^{-1}$']
handles = []

plt.figure(figsize=(8,6))
N_prompt_150_grid = griddata(points_os, N_prompt_150.values, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')
N_prompt_25_grid = griddata(points_os, N_prompt_25.values, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')



# Z_grid_14_os_30
contour1 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_30_mu, levels=[2], colors='darkorange')
#plt.clabel(contour1, fmt={2: fmt2}, inline=True, fontsize=10)
contour_mu30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30mup, levels=[2], colors='darkorange',linestyles='--')
contour_mu30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30mum, levels=[2], colors='darkorange',linestyles='--')
upper_segments_30 = contour_mu30p.allsegs[0]
lower_segments_30 = contour_mu30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#ffc680', alpha=0.5)

# Z_grid_14_os
contour2 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_mu, levels=[2], colors='navy')
#plt.clabel(contour2, fmt={2: fmt1}, inline=True, fontsize=10,manual=[(70,2e-3)])
contour_mu3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3mup, levels=[2], colors='navy',linestyles='--')
contour_mu3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3mum, levels=[2], colors='navy',linestyles='--')
upper_segments_3 = contour_mu3p.allsegs[0]
lower_segments_3 = contour_mu3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#8080c0', alpha=0.5)

#plt.title(r'ALP$\rightarrow \tau\tau \rightarrow e \mu $')
plt.ylim(1e-2,1e0)
plt.xlim(1,92)
plt.xscale('log')
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.axvline(x=mtau2, linewidth=1.5)
plt.yscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label=R'2$\sigma$ C.L. 25 ab$^{-1}$'),
    Line2D([0], [0], color='darkorange', lw=2, label=R'2$\sigma$ C.L. 150 ab$^{-1}$ '),
    Line2D([0], [0], color='navy', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='darkorange', lw=2, linestyle='--', label=R'150 ab$^{-1}$ 1$\sigma$ Error boundary')
]
plt.text(0.6, 0.7, R'$a \to \tau^{+}(\tau^{+}\to \mu^{+} \bar{v}_{\tau} v_{\mu}) \tau^{-}(\tau^{-}\to \mu^{-} v_{\tau} \bar{v}_{\mu})$', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
plt.legend(handles=legend_elements, loc='best', fontsize=10)
plt.savefig('./prompt/taumumu.png',dpi=300)
plt.show()

In [ ]:
from matplotlib.lines import Line2D
z_3_sum=(Z_grid_14_os_e + Z_grid_14_os_emu + Z_grid_14_os_mu)
z_30_sum=(Z_grid_14_os_30_e + Z_grid_14_os_30_emu + Z_grid_14_os_30_mu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)
std_3_e=sigma_Z(S_grid_14_os_e,background_14_os_e)
std_3_emu=sigma_Z(S_grid_14_os_emu,background_14_os_emu)
std_3_mu=sigma_Z(S_grid_14_os_mu,background_14_os_mu)
std2sum3=std_3_emu**2+std_3_e**2+std_3_mu**2

std_30_e=sigma_Z(S_grid_14_os_30_e,background_14_os_30_e)
std_30_emu=sigma_Z(S_grid_14_os_30_emu,background_14_os_30_emu)
std_30_mu=sigma_Z(S_grid_14_os_30_mu,background_14_os_30_mu)
std2sum30=std_30_emu**2+std_30_e**2+std_30_mu**2

sigma_tot3  = np.sqrt(std2sum3)
sigma_tot30 = np.sqrt(std2sum30)

# ----- 显著度：安全除法 ------------------------------
Z_3_comb  = np.divide(z_3_sum , sigma_tot3 ,
                      out=np.zeros_like(z_3_sum ), where=sigma_tot3  > 0)
Z_30_comb = np.divide(z_30_sum, sigma_tot30,
                      out=np.zeros_like(z_30_sum), where=sigma_tot30 > 0)

# ----- ±1σ 带 ----------------------------------------
z3p  = Z_3_comb  + sigma_tot3
z3m  = Z_3_comb  - sigma_tot3
z30p = Z_30_comb + sigma_tot30
z30m = Z_30_comb - sigma_tot30


plt.figure(figsize=(8, 6))

# Z_grid_14_os_30
contour_comb30 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_30_comb, levels=[2], colors='darkorange')
contour_comb30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30p, levels=[2], colors='darkorange',linestyles='--')
contour_comb30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30m, levels=[2], colors='darkorange',linestyles='--')
upper_segments_30 = contour_comb30p.allsegs[0]
lower_segments_30 = contour_comb30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#ffc680', alpha=0.5)



contour_comb3 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_3_comb, levels=[2], colors='navy')
contour_comb3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3p, levels=[2], colors='navy',linestyles='--')
contour_comb3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3m, levels=[2], colors='navy',linestyles='--')
upper_segments_3 = contour_comb3p.allsegs[0]
lower_segments_3 = contour_comb3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#8080c0', alpha=0.5)
#plt.clabel(contour_comb3, fmt={2: r'Z=2 (3fb$^{-1}$)'}, inline=True, fontsize=10,manual=[(40,2e-2)])
# plt.clabel(contour_comb3p, fmt={2: r'Zp=2 (3fb$^{-1}$)'}, inline=True, fontsize=10)
# plt.clabel(contour_comb3m, fmt={2: r'Zm=2 (3fb$^{-1}$)'}, inline=True, fontsize=10)
plt.xlim(1,92)
plt.ylim(1e-4,1e0)
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
#plt.axvline(x=mtau2, linewidth=1.5, label=R'$2m_{\tau}$')
plt.yscale('log')
plt.xscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label=R'25 ab$^{-1}$ 2$\sigma$ C.L. with $P_{prompt}$'),
    Line2D([0], [0], color='darkorange', lw=2, label=R'25 ab$^{-1}$ 2$\sigma$ C.L. with No-$P_{prompt}$'),
    Line2D([0], [0], color='navy', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='darkorange', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary')
]
plt.text(0.6, 0.7, R'combined $a \to \tau^{+}\tau^{-}$ channels', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
plt.legend(handles=legend_elements, loc='best', fontsize=10)
plt.savefig('./prompt/combined_tata.png',dpi=300)
plt.show()

In [ ]:
ma = [10, 30, 50, 70, 85]
S_eff = [0.964, 0.912, 0.849, 0.691, 0.314]
B_eff = [0.013, 0.013, 0.013, 0.013, 0.013]
B_cs = 84.29
mmu2=0.1057*2
# 读取数据
pho_ALP = pd.read_csv('./alp_decay_widths_had2.csv')
pho_ALP = pho_ALP[pho_ALP['ma'] >= mmu2].copy()
# 将数据类型转换为更小的浮动数类型以减少内存使用
pho_ALP_mumu = pho_ALP['xs_mumu_prompt'].astype('float32')
pho_ALP_mumu_no = pho_ALP['xs_mumu'].astype('float32')
pho_ALP_lambda = 1000 / pho_ALP['Lambda']

xs = pho_ALP_mumu
xs1 = pho_ALP_mumu_no
gstar = pho_ALP_lambda
MQ = pho_ALP['ma']

# 为信号截面插值，生成从7到79GeV每隔2GeV的数据点
#signal_xs_interp = np.interp(pho_ALP['ma'], ma, S_eff)
mq_14_os = np.logspace(np.log10(pho_ALP['ma'].min()), np.log10(pho_ALP['ma'].max()), 2000)
gstar_14_os = np.logspace(np.log10(gstar.min()), np.log10(gstar.max()), 200)
# 创建网格，减少网格密度以降低内存使用
#mq_14_os = np.linspace(pho_ALP['ma'].min(), pho_ALP['ma'].max(), 2000)
#gstar_14_os = np.linspace(gstar.min(), gstar.max(), 200)

# 创建网格
MQ_grid_14_os, Gstar_grid_14_os = np.meshgrid(mq_14_os, gstar_14_os)

# 构造插值用的点和目标点
points_frac = np.vstack((MQ, gstar)).T

# 使用二维插值来计算网格上的 S 和 Z
points_os = MQ,gstar.values
values_os = xs.values
values_os1 = xs1.values

# 对每个点进行信号和背景效率的插值
signal_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, S_eff)
background_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, B_eff)

# 将效率的插值结果重塑为(100, 100)形状
signal_efficiency_grid = signal_efficiency.reshape(MQ_grid_14_os.shape)
background_efficiency_grid = background_efficiency.reshape(MQ_grid_14_os.shape)


# 重新计算S_grid和背景
S_grid_14_os_amu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')  * L1*1000 * signal_efficiency_grid * 1000  # 25fb
background_14_os_amu = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000  # 25fb

# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_amu = S_grid_14_os_amu / np.sqrt(S_grid_14_os_amu + background_14_os_amu)

# 创建30fb情况
# 用30fb进行计算
S_grid_14_os_30_amu = griddata(points_os, values_os1, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')  * L1*1000 * signal_efficiency_grid * 1000  
background_14_os_30_amu = griddata(points_os, tuple([B_cs]*np.shape(values_os1)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000 


# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_30_amu = S_grid_14_os_30_amu / np.sqrt(S_grid_14_os_30_amu + background_14_os_30_amu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)

std_3_amu=sigma_Z(S_grid_14_os_amu,background_14_os_amu)
#Z_3_amu= Z_grid_14_os_amu / np.sqrt(std_3_amu)
z3amup = Z_grid_14_os_amu+np.sqrt(std_3_amu)
z3amum = Z_grid_14_os_amu-np.sqrt(std_3_amu)

std_30_amu=sigma_Z(S_grid_14_os_30_amu,background_14_os_30_amu)
#Z_30_amu= Z_grid_14_os_30_amu / np.sqrt(std_30_amu)
z30amup = Z_grid_14_os_30_amu+np.sqrt(std_30_amu)
z30amum = Z_grid_14_os_30_amu-np.sqrt(std_30_amu)

# 绘制单张图包含两个等值线图
plt.figure(figsize=(8, 6))

# Z_grid_14_os_30 150ab
contour1 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_30_amu, levels=[2], colors='darkorange')
contour_amu30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30amup, levels=[2], colors='darkorange',linestyles='--')
contour_amu30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30amum, levels=[2], colors='darkorange',linestyles='--')
upper_segments_30 = contour_amu30p.allsegs[0]
lower_segments_30 = contour_amu30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#ffc680', alpha=0.5)

# Z_grid_14_os
contour2 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_amu, levels=[2], colors='navy')
#plt.clabel(contour2, fmt={2: fmt1}, inline=True, fontsize=10,manual=[(70,2e-3)])
contour_amu3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3amup, levels=[2], colors='navy',linestyles='--')
contour_amu3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3amum, levels=[2], colors='navy',linestyles='--')
upper_segments_3 = contour_amu3p.allsegs[0]
lower_segments_3 = contour_amu3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#8080c0', alpha=0.5)

#plt.title(r'ALP$\rightarrow \tau\tau \rightarrow e \mu $')
plt.ylim(1e-3,1e1)
plt.xlim(0.1,92)
#plt.axvline(x=mmu2, linewidth=1.5)
#plt.axvline(x=mtau2, linewidth=1.5)
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.yscale('log')
plt.xscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label=R'25 ab$^{-1}$ 2$\sigma$ C.L. with $P_{prompt}$'),
    Line2D([0], [0], color='darkorange', lw=2, label=R'25 ab$^{-1}$ 2$\sigma$ C.L. with No-$P_{prompt}$'),
    Line2D([0], [0], color='navy', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='darkorange', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary') 
]
plt.text(0.5, 0.7, R'$a \to \mu^{+} \mu^{-}$', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
plt.legend(handles=legend_elements, loc='best', fontsize=10)
plt.savefig('./prompt/amumu.png',dpi=300)
plt.show()
